# Task 1: MERT vs CultureMERT for Raga Classification

Run this notebook in **Google Colab with GPU enabled**. It expects `task1_raga_classification_colab.zip` to be uploaded in the first code cell.

## 1. Upload Project Zip

Upload `task1_raga_classification_colab.zip` when prompted.

In [ ]:
from google.colab import files
uploaded = files.upload()
!rm -rf /content/task1_raga_classification
!unzip -q task1_raga_classification_colab.zip -d /content/
%cd /content/task1_raga_classification
!find . -maxdepth 2 -type f | sort | head -80

## 2. Install Dependencies

Colab usually already has a GPU build of PyTorch, so this cell installs the rest without replacing torch.

In [ ]:
!pip -q install "transformers>=4.35" "librosa>=0.10" soundfile numpy pandas scikit-learn matplotlib seaborn umap-learn tensorboard tqdm pyyaml mirdata joblib nnAudio
!python scripts/00_check_env.py

## 3. Download and Inspect Saraga

In [ ]:
!python scripts/01_download_saraga.py
!python scripts/02_explore_dataset.py

## 4. Preprocess Audio and Create Splits

This creates 24 kHz, mono, 10-second clips and splits by original track ID.

In [ ]:
!python scripts/03_preprocess.py
!python scripts/04_create_splits.py

## 5. Extract Embeddings

This is the first expensive part. On a T4 GPU, run MERT first, then CultureMERT.

In [ ]:
!python scripts/05_extract_embeddings.py --model mert_95m
!python scripts/05_extract_embeddings.py --model culturemert_95m

## 6. Visualize and Train Linear Probes

In [ ]:
!python scripts/06_visualize_embeddings.py
!python scripts/07_train_probe.py
!python scripts/09_evaluate.py

## 7. Fine-Tuning: Start with Frozen Backbones

These two are the safest fine-tuning experiments. Run partial/full only if GPU memory and time are okay.

In [ ]:
!python scripts/08_finetune.py --config configs/experiment_configs.yaml --experiment mert_95m_frozen
!python scripts/08_finetune.py --config configs/experiment_configs.yaml --experiment culturemert_95m_frozen
!python scripts/09_evaluate.py

## 8. Optional Fine-Tuning: Partial and Full

Run this only after the frozen experiments finish.

In [ ]:
!python scripts/08_finetune.py --config configs/experiment_configs.yaml --experiment mert_95m_partial_unfreeze
!python scripts/08_finetune.py --config configs/experiment_configs.yaml --experiment mert_95m_full_finetune
!python scripts/08_finetune.py --config configs/experiment_configs.yaml --experiment culturemert_95m_partial_unfreeze
!python scripts/08_finetune.py --config configs/experiment_configs.yaml --experiment culturemert_95m_full_finetune
!python scripts/09_evaluate.py

## 9. Download Results

In [ ]:
!zip -qr /content/task1_raga_results.zip results models/probes notes README.md
files.download('/content/task1_raga_results.zip')